# **https://www.kaggle.com/competitions/home-credit-default-risk/data**
# **https://www.kaggle.com/code/carlosdg/xgboost-with-scikit-learn-pipeline-gridsearchcv**

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
REPO_DIR="/content/drive/MyDrive/HomeCredit"
%cd $REPO_DIR

/content/drive/MyDrive/HomeCredit


In [16]:
import sys, os
sys.path.insert(0, os.getcwd())

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

### CLEAN CODE WITH DATA MERGE AND DATA CLEANING

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load datasets
# Note: Ensure these files are in your current working directory or specify the path
app_train = pd.read_csv('application_train.csv')
app_test = pd.read_csv('application_test.csv')
bureau = pd.read_csv('bureau.csv')
prev_app = pd.read_csv('previous_application.csv')


# Aggregate Bureau data (Example: count of previous loans)
bureau_counts = bureau.groupby('SK_ID_CURR', as_index=False)['SK_ID_BUREAU'].count().rename(columns={'SK_ID_BUREAU': 'BUREAU_LOAN_COUNT'})

# # Aggregate Previous Applications (Example: average previous credit amount)
#prev_agg = prev_app.groupby('SK_ID_CURR', as_index=False)['AMT_CREDIT'].agg(['mean']).reset_index()
#prev_agg.columns = ['SK_ID_CURR', 'PREV_APP_CREDIT_MEAN']


# Removing .reset_index() because as_index=False already handled it
prev_agg = prev_app.groupby('SK_ID_CURR', as_index=False)['AMT_CREDIT'].agg(['mean'])

# This will now work because prev_agg has exactly 2 columns: ['SK_ID_CURR', 'mean']
prev_agg.columns = ['SK_ID_CURR', 'PREV_APP_CREDIT_MEAN']

# Merge into Main Dataframes
def merge_supplementary(df):
    df = df.merge(bureau_counts, on='SK_ID_CURR', how='left')
    df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
    # Fill count NAs with 0
    df['BUREAU_LOAN_COUNT'] = df['BUREAU_LOAN_COUNT'].fillna(0)
    return df

app_train = merge_supplementary(app_train)
app_test = merge_supplementary(app_test)



In [20]:
#Identify columns with more than 50% missing data
missing_stats = app_train.isnull().mean()
cols_to_drop = missing_stats[missing_stats > 0.5].index.tolist()

# Drop from both sets
app_train.drop(columns=cols_to_drop, inplace=True)
app_test.drop(columns=cols_to_drop, inplace=True)


numeric_cols = app_train.select_dtypes(include=[np.number]).columns
app_train[numeric_cols] = app_train[numeric_cols].fillna(app_train[numeric_cols].median())
app_test[numeric_cols.drop('TARGET', errors='ignore')] = app_test[numeric_cols.drop('TARGET', errors='ignore')].fillna(app_train[numeric_cols].median())

#Final Imputation (Simple median fill for remaining NAs)
#app_train = app_train.fillna(app_train.median())


In [21]:
print(missing_stats)

SK_ID_CURR                    0.000000
TARGET                        0.000000
NAME_CONTRACT_TYPE            0.000000
CODE_GENDER                   0.000000
FLAG_OWN_CAR                  0.000000
                                ...   
AMT_REQ_CREDIT_BUREAU_MON     0.135016
AMT_REQ_CREDIT_BUREAU_QRT     0.135016
AMT_REQ_CREDIT_BUREAU_YEAR    0.135016
BUREAU_LOAN_COUNT             0.000000
PREV_APP_CREDIT_MEAN          0.053507
Length: 124, dtype: float64


In [22]:
# 4. Handle Categorical Data (One-Hot Encoding)
# This converts strings to numbers so the model can read them
app_train = pd.get_dummies(app_train)
app_test = pd.get_dummies(app_test)

# Align columns (ensures train and test have the same features)
train_labels = app_train['TARGET']
app_train, app_test = app_train.align(app_test, join='inner', axis=1)
app_train['TARGET'] = train_labels

In [23]:
# Train-Test Split
X = app_train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app_train['TARGET']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Cleaned Training shape: {X_train.shape}")
print(f"Cleaned Validation shape: {X_val.shape}")

Cleaned Training shape: (246008, 191)
Cleaned Validation shape: (61503, 191)


In [13]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# 1. Calculate class imbalance ratio for scale_pos_weight
# Formula: count(negative) / count(positive)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# 2. Initialize the XGBClassifier
# We use 'binary:logistic' for binary classification
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=ratio,  # Handles class imbalance
    early_stopping_rounds=20,
    use_label_encoder=False,
    eval_metric='auc'        # Primary metric for this competition
)

# 3. Fit the model with early stopping to prevent overfitting
print("Starting XGBoost training...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# 4. Predict probabilities (Competition usually requires ROC-AUC on probabilities)
y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
y_pred = xgb_model.predict(X_val)

# 5. Evaluate Performance
auc_score = roc_auc_score(y_val, y_pred_proba)
print(f"\nValidation ROC-AUC Score: {auc_score:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

Starting XGBoost training...


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [14:58:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	validation_0-auc:0.71487
[50]	validation_0-auc:0.74568
[100]	validation_0-auc:0.75251
[150]	validation_0-auc:0.75582
[200]	validation_0-auc:0.75696
[250]	validation_0-auc:0.75764
[300]	validation_0-auc:0.75784
[301]	validation_0-auc:0.75788

Validation ROC-AUC Score: 0.7579

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.73      0.83     56538
           1       0.17      0.65      0.27      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.69      0.55     61503
weighted avg       0.90      0.72      0.78     61503



In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

# 1. Define the parameter distributions to sample from
param_dist = {
    'max_depth': [3, 4, 5, 6, 8, 10],            # Common range for imbalanced data
    'min_child_weight': [1, 3, 5, 7],           # Lower values help capture minority groups
    'learning_rate': [0.01, 0.05, 0.1],         # Smaller rates usually perform better
    'subsample': [0.6, 0.7, 0.8, 0.9],          # Prevents overfitting
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],   # Feature sampling
    'reg_alpha': [0, 0.1, 0.5, 1],              # L1 regularization
    'reg_lambda': [1, 1.5, 2, 5]                # L2 regularization
}

# 2. Initialize the base model with constant parameters
# Note: n_estimators is set high because early_stopping will handle the actual count
base_xgb = XGBClassifier(
    n_estimators=1000,
    scale_pos_weight=ratio,      # Uses the ratio we calculated earlier
    early_stopping_rounds=50,    # Stops if no improvement in 50 rounds
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

# 3. Setup the RandomizedSearchCV
# n_iter=20 means it will randomly pick 20 combinations to test
random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',           # Primary competition metric
    cv=3,                        # 3-fold cross-validation to save time
    verbose=2,
    random_state=42,
    return_train_score=True
)

# 4. Run the search
# We must provide eval_set here for early stopping to work
print("Starting hyperparameter search...")
random_search.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# 5. Review results
print(f"\nBest Score: {random_search.best_score_:.4f}")
print(f"Best Parameters: {random_search.best_params_}")

Starting hyperparameter search...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
# 1. Combine training and validation sets for the final run
# This ensures the model learns from every available labeled example
X_final_train = pd.concat([X_train, X_val])
y_final_train = pd.concat([y_train, y_val])

# 2. Use the best parameters found during RandomizedSearch
# You can reference random_search.best_params_ directly
best_params = random_search.best_params_

# 3. Initialize the final model
# Note: We remove 'early_stopping_rounds' here because we no longer have a
# separate validation set to monitor. We use the 'n_estimators' found by the search.
final_model = XGBClassifier(
    **best_params,
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=ratio,
    eval_metric='auc'
)

print("Fitting final model on the full dataset...")
final_model.fit(X_final_train, y_final_train)

# 4. Prepare the Test Data
# Ensure the test set features match the training set exactly
X_test = app_test.drop(columns=['SK_ID_CURR'])

# 5. Generate Predictions (Probabilities)
# In Credit Risk, we need the probability of default (class 1)
print("Generating test predictions...")
test_probs = final_model.predict_proba(X_test)[:, 1]

# 6. Create the Submission File
submission = pd.DataFrame({
    'SK_ID_CURR': app_test['SK_ID_CURR'],
    'TARGET': test_probs
})

submission.to_csv('home_credit_final_submission.csv', index=False)
print("Submission file 'home_credit_final_submission.csv' is ready!")